In [0]:
# Since we cannot turn off THE BROADCAST HASH JOIN (Because data size is small, it can automatically perform broadcast join)
# We will use "hints" rather than trying to change the global configuration in this Databricks environment.

"""
# Prevent broadcast
df1.join(
    df2.hint("NO_BROADCAST_HASH"),
    "id",
    "left"
).explain()
"""

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [id#11253L, name#11254, salary#11256L]
         +- PhotonBroadcastHashJoin [id#11253L], [id#11255L], Inner, BuildRight, false, true
            :- PhotonRowToColumnar
            :  +- LocalTableScan [id#11253L, name#11254]
            +- PhotonShuffleExchangeSource
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#7267]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonRowToColumnar
                        +- LocalTableScan [id#11255L, salary#11256L]


== Photon Explanation ==
The query is fully supported by Photon.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast 

# Create first dataframe
data1 = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charles"),
    (4, "David"),
    (5, "Eva")
]
df1 = spark.createDataFrame(data1, ["id", "name"])

# Create second dataframe
data2 = [
    (1, 50000),
    (2, 60000),
    (3, 70000),
    (6, 80000)
]
df2 = spark.createDataFrame(data2, ["id", "salary"])

## LEFT JOIN (By SORT MERGE JOIN)

In [0]:
df_left_merge = df1.join(
    df2.hint("MERGE"),               # Explicit SORT MERGE join
    "id",
    "left"
)

In [0]:
df_left_merge.show()

df_left_merge.explain()

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  1|  Alice| 50000|
|  2|    Bob| 60000|
|  3|Charles| 70000|
|  4|  David|  NULL|
|  5|    Eva|  NULL|
+---+-------+------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   Project [id#12100L, name#12101, salary#12103L]
   +- SortMergeJoin [id#12100L], [id#12102L], LeftOuter
      :- ColumnarToRow
      :  +- PhotonResultStage
      :     +- PhotonSort [id#12100L ASC NULLS FIRST]
      :        +- PhotonShuffleExchangeSource
      :           +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#10537]
      :              +- PhotonShuffleExchangeSink hashpartitioning(id#12100L, 16)
      :                 +- PhotonRowToColumnar
      :                    +- LocalTableScan [id#12100L, name#12101]
      +- ColumnarToRow
         +- PhotonResultStage
            +- PhotonSort [id#12102L ASC NULLS FIRST]
               +- PhotonShuffleExchangeSource
                  +- PhotonShuffleMapStage ENS

## LEFT JOIN (By BROADCAST HASH JOIN)

In [0]:
df_left_broadcast = df1.join(
    df2.hint("BROADCAST"),
    "id",
    "left"
)

In [0]:
df_left_broadcast.show()

df_left_broadcast.explain()

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  1|  Alice| 50000|
|  2|    Bob| 60000|
|  3|Charles| 70000|
|  4|  David|  NULL|
|  5|    Eva|  NULL|
+---+-------+------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [id#12132L, name#12133, salary#12135L]
         +- PhotonBroadcastHashJoin [id#12132L], [id#12134L], LeftOuter, BuildRight, false, true
            :- PhotonRowToColumnar
            :  +- LocalTableScan [id#12132L, name#12133]
            +- PhotonShuffleExchangeSource
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#10708]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonRowToColumnar
                        +- LocalTableScan [id#12134L, salary#12135L]


== Photon Explanation ==
The query is fully supported by Photon.


%md
## LEFT JOIN (By SHUFFLE HASH JOIN)

In [0]:
df_left_shuffle = df1.join(
    df2.hint("SHUFFLE_HASH"),
    "id",
    "left"
)

In [0]:
df_left_shuffle.show()

df_left_shuffle.explain()

+---+-------+------+
| id|   name|salary|
+---+-------+------+
|  1|  Alice| 50000|
|  2|    Bob| 60000|
|  3|Charles| 70000|
|  4|  David|  NULL|
|  5|    Eva|  NULL|
+---+-------+------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [id#11235L, name#11236, salary#11238L]
         +- PhotonShuffledHashJoin [id#11235L], [id#11237L], LeftOuter, BuildRight
            :- PhotonShuffleExchangeSource
            :  +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7397]
            :     +- PhotonShuffleExchangeSink hashpartitioning(id#11235L, 16)
            :        +- PhotonRowToColumnar
            :           +- LocalTableScan [id#11235L, name#11236]
            +- PhotonShuffleExchangeSource
               +- PhotonShuffleMapStage ENSURE_REQUIREMENTS, [id=#7402]
                  +- PhotonShuffleExchangeSink hashpartitioning(id#11237L, 16)
                     +- PhotonRowToColumna